# Crystal Eye | ACME Hands-on in Python

**Crystal Eye tutorial team · ACME Hands-on**

- **Generate:** choose a detector-frame direction and inspect the expected counts per pixel.
- **Detect:** estimate the background from the first 8 seconds and look for a burst in the residuals.
- **Localize:** compare the pixel pattern with response templates and measure the angular separation.

Run the cells from top to bottom. On the web page choose **Enable Run buttons** first. In JupyterLab, select a code cell and press **Shift + Enter**. The Python implementations below use PyROOT to read the detector inputs. Plots appear when you run the exercise cells.

## Setup

Run this cell to load PyROOT and check the detector inputs. The setup and function cells print a confirmation. The exercise cells below produce the plots.

In [ ]:
import os
from pathlib import Path
from array import array
from dataclasses import dataclass
from io import BytesIO
from uuid import uuid4

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
import ROOT

source = Path(os.environ.get("ACME_TUTORIAL_DIR", "../../ACMETutorial")).expanduser().resolve()
required = ("allBackground.root", "allEffArea.root", "templates5deg.root",
            "temEdepPix2degNew.root", "upCal.txt")
missing = [name for name in required if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"Set ACME_TUTORIAL_DIR to the shared input directory. Missing: {missing}")

from matplotlib.ticker import MaxNLocator, StrMethodFormatter

INK, MUTED = "#111827", "#4B5563"
TEAL, BLUE, AMBER = "#0F766E", "#1D4ED8", "#B45309"
BASELINE, GRID = "#6B7280", "#D1D5DB"

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "font.family": "DejaVu Sans", "font.size": 10,
    "text.color": INK, "axes.labelcolor": MUTED, "axes.titlecolor": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#9CA3AF", "axes.linewidth": 0.8,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "legend.frameon": False,
})

def figure_header(fig, title, subtitle, note):
    fig.text(.10, .95, title, ha="left", va="top", fontsize=15, weight="bold")
    fig.text(.10, .885, subtitle, ha="left", va="top", color=MUTED, fontsize=10)
    fig.text(.10, .035, note, ha="left", va="bottom", color="#6B7280", fontsize=8.5)

def style_counts(ax):
    ax.set_axisbelow(True)
    ax.grid(axis="y", linestyle="--", color=GRID, alpha=.55)
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))

def show_plot(fig):
    png = BytesIO()
    fig.savefig(png, format="png", dpi=150, bbox_inches="tight", pad_inches=.15)
    plt.close(fig)
    display(Image(data=png.getvalue()))

print(f"Setup complete. Found {len(required)} detector inputs. Run the detector helpers next.")

## Detector inputs

The notebook uses the supplied Geant4 reference products. The [detector chapter](../theory/detector.md) describes the geometry, and the [response chapter](../theory/response.md) explains how the template banks were built.

### Read the detector input files

Run these helpers before the three exercise steps. They open ROOT files, look up histograms and find nearby direction templates.

In [ ]:
def root_object(file, name, kind):
    obj = file.Get(name)
    if not obj or not obj.InheritsFrom(kind):
        raise ValueError(f"{file.GetName()}: expected {kind} named {name}")
    return obj


def root_input(folder, filename):
    return ROOT.TFile.Open(str(Path(folder) / filename), "READ")


def detached_clone(histogram):
    copy = histogram.Clone(uuid4().hex)
    copy.SetDirectory(0)
    return copy


def direction(theta, phi):
    result = ROOT.TVector3()
    result.SetMagThetaPhi(1, np.deg2rad(theta), np.deg2rad(phi))
    return result


def angles(vec):
    return float(np.rad2deg(vec.Theta())), float(np.rad2deg(vec.Phi()) % 360)


def template_positions(file):
    tree = root_object(file, "tLocInfo", "TTree")
    return [(float(row.theta), float(row.phi)) for row in tree]


def template_weights(positions, theta, phi):
    requested = direction(theta, phi)
    nearby = sorted((np.float32(np.rad2deg(requested.Angle(direction(t, p)))), t, p)
                    for t, p in positions)
    if nearby[0][0] == 0:
        return [(nearby[0][1], nearby[0][2], np.float32(1))]
    a, b, c = [direction(t, p) for _, t, p in nearby[:3]]
    count = 2 if abs(a.Dot(b.Cross(c))) < 1e-9 else 3
    nearest = nearby[:count]
    weights = [np.float32(1.0 / float(distance)) for distance, _, _ in nearest]
    total = sum(weights, np.float32(0))
    return [(t, p, weight / total) for (_, t, p), weight in zip(nearest, weights)]

print("Detector helpers ready. Run the burst generator definition next.")

## 1. Generate a burst

Choose detector-frame zenith and azimuth angles in degrees, and the parameters of the cut-off power law. The code selects neighboring response templates and estimates the energy spectrum and counts per pixel. First run the supplied example. Then try an angle of 45° or reduce the amplitude by a factor of ten.

The first plot shows how the expected source count rate varies across detector pixels for your settings.

In [ ]:
if "template_weights" not in globals() or "show_plot" not in globals():
    print("Run Setup and the detector helpers first.")
else:
    @dataclass
    class SourceResponse:
        pixels: object
        energy: object
        templates: list


    def generate_grb(folder, theta=0, phi=0, amplitude=0.026, index=-1.07, cutoff_kev=756.4):
        """Port CEGenGRB: combine template event prefixes and apply a cut-off power law."""
        if (not np.isfinite([theta, phi, amplitude, index, cutoff_kev]).all()
                or not 0 <= theta <= 115 or not 0 <= phi < 360 or amplitude <= 0 or cutoff_kev <= 0):
            raise ValueError("Use 0 ≤ theta ≤ 115, 0 ≤ phi < 360, and positive amplitude and cutoff.")

        low = np.float32(np.log10(30))
        step = (np.float32(5) - low) / np.float32(100)
        edges = array("f", [10.0 ** float(low + np.float32(i) * step) for i in range(101)])
        pixels = ROOT.TH1F(uuid4().hex, "", 126, 0, 126)
        energy = ROOT.TH1F(uuid4().hex, "", 100, edges)
        pixels.SetDirectory(0)
        energy.SetDirectory(0)
        spectrum = ROOT.TF1(uuid4().hex, "[0]*((x/100)**[1])*exp(-([1]+2)*x/[2])", 20, 1e6)
        spectrum.SetParameters(*[float(np.float32(x)) for x in (amplitude, index, cutoff_kev)])

        selected = []
        with root_input(folder, "templates5deg.root") as templates:
            weighted = template_weights(template_positions(templates), np.float32(theta), np.float32(phi))
            suffixes = [(f"{int(t)}_{int(p)}", weight) for t, p, weight in weighted]
            primary = [root_object(templates, f"hPeng_{suffix}", "TH1F") for suffix, _ in suffixes]
            if len({hist.GetEntries() for hist in primary}) != 1:
                raise ValueError("Primary-energy template entry counts differ.")
            mean = detached_clone(primary[0])
            mean.Reset()
            for hist, (_, weight) in zip(primary, suffixes):
                mean.Add(hist, float(weight))

            rate = np.zeros(mean.GetNbinsX() + 2, dtype=np.float32)
            for i in range(1, mean.GetNbinsX() + 1):
                count = mean.GetBinContent(i)
                if count:
                    low_edge = mean.GetBinLowEdge(i)
                    rate[i] = 1296 * spectrum.Integral(low_edge, low_edge + mean.GetBinWidth(i)) / count

            # Branch names contain literal brackets in the supplied ROOT trees.
            for (t, p, weight), (suffix, _) in zip(weighted, suffixes):
                tree = root_object(templates, f"tEdepPix_{suffix}", "TTree")
                entries = int(np.float32(tree.GetEntries()) * weight)
                selected.append((t, p, float(weight), entries))
                tree.SetBranchStatus("*", 0)
                for name in ("peng", "totEdep", "pixID[px]", "pixEdep[px]"):
                    tree.SetBranchStatus(name, 1)
                for i in range(entries):
                    tree.GetEntry(i)
                    deposited = tree.totEdep
                    norm = float(rate[mean.FindBin(tree.peng)])
                    ids, deposits = getattr(tree, "pixID[px]"), getattr(tree, "pixEdep[px]")
                    if len(ids) != len(deposits) or (len(ids) and deposited <= 0):
                        raise ValueError(f"Invalid energy deposit in {suffix} event {i}.")
                    for pixel, deposit in zip(ids, deposits):
                        fraction = float(np.float32(deposit) / np.float32(deposited))
                        pixels.Fill(pixel, fraction * norm)
                    energy.Fill(deposited, norm)
        return SourceResponse(pixels, energy, selected)
    print("Burst generator ready. Set the parameters and run the next cell.")

Run the cell below to set the burst parameters and plot the expected counts per pixel. Change one parameter and rerun it to compare responses.

In [ ]:
theta_deg, phi_deg = 0, 0
amplitude, spectral_index, cutoff_kev = 0.026, -1.07, 756.4

response = None
if "generate_grb" not in globals():
    print("Run Setup, the detector helpers and the burst generator definition first.")
else:
    try:
        response = generate_grb(source, theta_deg, phi_deg, amplitude, spectral_index, cutoff_kev)
    except (ValueError, TypeError) as error:
        print(f"Could not generate the burst: {error}")
if response is not None:
    print(f"Expected total detector rate: {response.energy.Integral():.2f} counts/s")
    for theta, phi, weight, events in response.templates:
        print(f"Template θ={theta:.2f}°, φ={phi:.2f}° · weight {weight:.3f} · {events:,} events")

    fig, ax = plt.subplots(figsize=(9.2, 4.0))
    fig.subplots_adjust(left=.10, right=.975, top=.79, bottom=.22)
    figure_header(
        fig, "The expected burst response varies across pixels",
        f"Detector direction: θ = {theta_deg:g}°, φ = {phi_deg:g}° · expected source counts per second",
        "Source: Crystal Eye generator templates · simulated source contribution.",
    )
    pixel = range(response.pixels.GetNbinsX())
    counts_per_pixel = [response.pixels.GetBinContent(i + 1) for i in pixel]
    ax.step(pixel, counts_per_pixel, where="mid", color=TEAL, linewidth=1.45)
    ax.set(xlabel="Detector pixel ID", ylabel="Expected counts/s", xlim=(0, len(pixel) - 1))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    style_counts(ax)
    show_plot(fig)

## 2. Detection: when did the counts rise?
The Python simulation generates 250 ms count bins with Gaussian fluctuations. The top plot compares simulated total counts to the model background mean and its counting spread. The lower plot subtracts the mean estimated from the first 8 s. Look for an excess before revealing the expected injected signal. Change the amplitude in section 1, then rerun sections 1 and 2 with the same seed.

The program also reports its original experimental trigger. Its threshold uses the dispersion of the 8 s aggregate and compares it with a single-bin excess. Treat it as program output rather than a calibrated significance.

In [ ]:
def lightcurve(folder, response, seed=2026):
    """Port CELightcurve's seeded Gaussian light curve and legacy trigger."""
    if type(seed) is not int or not 0 < seed < 2**32:
        raise ValueError("Choose a nonzero 32-bit integer seed.")
    with root_input(folder, "allBackground.root") as file:
        background_rate = np.float32(root_object(file, "hTotBkg", "TH1F").Integral())
    width, window, span, onset = 0.25, 8.0, 30.0, 10.0
    pulse = ROOT.TF1(uuid4().hex, lambda t, _: (
        np.exp(1 - 0.5 / (t[0] - onset) - (t[0] - onset) / 2) if t[0] > onset else 0
    ), -window, span, 0)
    source_rate = response.energy.Integral()
    expected_background = float(background_rate) * width
    random = ROOT.TRandom3(seed)
    starts = np.arange(0, span, width)
    expected_source = np.empty_like(starts)
    observed = np.empty(starts.size, dtype=np.float32)
    for i, start in enumerate(starts):
        expected_source[i] = source_rate * pulse.Integral(start, start + width)
        mean = expected_background + expected_source[i]
        observed[i] = random.Gaus(mean, np.sqrt(mean))
        # The original draws a separate overlapping-window sample on each step.
        integrated = background_rate * window + source_rate * pulse.Integral(start + width - window, start + width)
        random.Gaus(integrated, np.sqrt(integrated))

    bins = int(window / width)
    consecutive = 0
    triggers = []
    for i in range(bins, len(observed)):
        if not consecutive:
            past = np.float32(np.sum(observed[i - bins:i], dtype=np.float64))
            threshold = 3 * np.sqrt(past)
            trigger_start = starts[i]
        excess = observed[i] - past / bins
        consecutive = consecutive + 1 if excess > threshold else 0
        if consecutive == 3:
            triggers.append(float(trigger_start))
    return starts + width / 2, observed, expected_background, expected_source, triggers

print("Detection function ready. Run the next cell to simulate the counts.")

Run the simulation with the default random seed, then plot the counts and residuals in the next cell.

In [ ]:
seed = 2026
curve_response = None
if globals().get("response") is None:
    print("Run the burst parameter cell in section 1 first.")
elif "lightcurve" not in globals():
    print("Run the detection function definition above first.")
else:
    try:
        time_s, total, background, injected_signal, triggers = lightcurve(source, response, seed)
    except ValueError as error:
        print(f"Could not simulate the counts: {error}")
    else:
        curve_response = response
        print(f"Simulated {len(time_s)} bins. Legacy trigger onsets (s): {triggers or 'none'}")
        print("Run the next cell to plot the counts and residuals.")

In [ ]:
reveal_injected = False

if globals().get("curve_response") is None or curve_response is not globals().get("response"):
    print("Run the count simulation cell above for the current burst first.")
else:
    baseline_window = time_s < 8
    estimated_background = total[baseline_window].mean()
    fig, (ax, residual) = plt.subplots(2, 1, sharex=True, figsize=(10, 6.7))
    fig.subplots_adjust(left=.10, right=.975, top=.79, bottom=.16, hspace=.37)
    figure_header(
        fig, "Does an excess stand out above the background?",
        "30-second simulation · 250 ms bins · background estimated from the first 8 seconds",
        "Source: Crystal Eye simulated light curve · shaded spread is ±√mean, not a trigger threshold.",
    )
    ax.step(time_s, total, where="mid", color=BLUE, linewidth=1.35)
    ax.axhline(background, color=BASELINE, linestyle="--", linewidth=1)
    ax.fill_between(time_s, max(0, background - np.sqrt(background)),
                    background + np.sqrt(background), color=GRID, alpha=.48)
    ax.set_title("Total counts and expected background", loc="left", fontsize=11, weight="bold")
    ax.set_ylabel("Counts/bin")
    residual.step(time_s, total - estimated_background, where="mid", color=TEAL, linewidth=1.35)
    residual.axhline(0, color=BASELINE, linewidth=.8)
    residual.axvspan(0, 8, color=GRID, alpha=.26)
    if reveal_injected:
        residual.step(time_s, injected_signal, where="mid", color=AMBER, linestyle="--", linewidth=1.55)
    residual.set_title(
        "Counts above or below the estimated mean" + (" · dashed: injected GRB" if reveal_injected else ""),
        loc="left", fontsize=11, weight="bold",
    )
    residual.set(xlabel="Time (s)", ylabel="Excess counts/bin", xlim=(0, 30))
    for chart in (ax, residual):
        style_counts(chart)
    show_plot(fig)

## 3. Localization: where did the burst come from?

Use the expected source pixel counts to compute a detector centroid. The centroid selects an effective area. The code compares the pixel distributions to the direction templates with ROOT's Kolmogorov–Smirnov histogram test, then averages retained directions with the KS scores as weights.

This exercise uses the simulated source response. The noisy light curve is a separate detection exercise. The scores compare templates and do not define a confidence region.

The polar plot shows the injected and reconstructed directions alongside the retained templates. Radius is detector zenith and angle around the plot is detector azimuth.

In [ ]:
if "template_positions" not in globals() or "show_plot" not in globals():
    print("Run Setup and the detector helpers first.")
else:
    @dataclass
    class Localization:
        theta: float
        phi: float
        centroid_theta: float
        centroid_phi: float
        scores: np.ndarray
        retained: np.ndarray


    def localize(folder, response):
        """Port CELoc: use the ROOT histogram KS score, then average retained directions."""
        coords = np.loadtxt(Path(folder) / "upCal.txt", dtype=np.float32)
        pixel_hist = response.pixels
        weights = np.array([pixel_hist.GetBinContent(i + 1) for i in range(len(coords))])
        if not np.isfinite(weights).all() or weights.sum() <= 0:
            raise ValueError("No usable source counts for localization.")
        centroid = ROOT.TVector3(*(weights @ coords[:, 1:]))
        centroid_theta, centroid_phi = angles(centroid)
        energy = detached_clone(response.energy)
        with root_input(folder, "allEffArea.root") as file:
            effective = root_object(file, f"hEffArea{int(min(90, centroid_theta))}", "TH1F")
            energy.Divide(effective)
        ny_energy = energy.GetNbinsX()
        for j in range(1, ny_energy + 1):
            energy.SetBinContent(j, energy.GetBinContent(j) / energy.GetBinWidth(j))
        flux = np.array([energy.GetBinContent(j) for j in range(ny_energy + 2)])
        errors = [pixel_hist.GetBinError(i + 1) for i in range(pixel_hist.GetNbinsX())]

        scores = []
        with root_input(folder, "temEdepPix2degNew.root") as templates:
            for t, p in template_positions(templates):
                hist = root_object(templates, f"hEdepPix_{int(t)}_{int(p)}", "TH2F")
                nx, ny = hist.GetNbinsX(), hist.GetNbinsY()
                if ny != ny_energy or nx != len(errors):
                    raise ValueError("Template dimensions do not match the sample response.")
                # ROOT stores X fastest. ProjectionX includes the Y flow bins.
                storage = np.frombuffer(hist.GetArray(), dtype=np.float32, count=hist.GetNcells())
                storage = storage.reshape(ny + 2, nx + 2)
                storage[1:-1, 1:-1] *= flux[1:-1, None]
                projection = hist.ProjectionX(uuid4().hex)
                projection.SetDirectory(0)
                ROOT.SetOwnership(projection, True)
                for i, error in enumerate(errors, 1):
                    projection.SetBinError(i, error)
                scores.append((t, p, np.float32(pixel_hist.KolmogorovTest(projection))))
                del storage, projection
                # Release each loaded histogram instead of keeping 2° templates in memory.
                hist.SetDirectory(0)
                ROOT.SetOwnership(hist, True)
                del hist

        scores = np.array(scores)
        if not len(scores) or not np.isfinite(scores[:, 2]).all() or scores[:, 2].max() <= 0:
            raise ValueError("No comparable localization templates.")
        # Apply the original relative cut without erasing from the vector during traversal.
        retained = scores[:, 2] >= 0.01 * scores[:, 2].max()
        result = ROOT.TVector3()
        for t, p, score in scores[retained]:
            result += score * direction(t, p)
        if result.Mag2() == 0:
            raise ValueError("The weighted direction is degenerate.")
        theta, phi = angles(result)
        return Localization(theta, phi, centroid_theta, centroid_phi, scores, retained)
    print("Localization function ready. Run the next cell to estimate the direction.")

Run the cell below to compare the injected direction with the detector centroid and the direction reconstructed from the templates.

In [ ]:
if globals().get("response") is None:
    print("Run the burst parameter cell in section 1 first.")
elif "localize" not in globals():
    print("Run the localization function definition above first.")
else:
    result = localize(source, response)
    recovered_theta, recovered_phi = result.theta, result.phi

    def unit_vector(theta, phi):
        t, p = np.deg2rad([theta, phi])
        return np.array([np.sin(t) * np.cos(p), np.sin(t) * np.sin(p), np.cos(t)])

    injected = unit_vector(theta_deg, phi_deg)
    recovered = unit_vector(recovered_theta, recovered_phi)
    separation = np.rad2deg(np.arctan2(np.linalg.norm(np.cross(injected, recovered)), injected @ recovered))
    print(f"Centroid θ={result.centroid_theta:.2f}°, φ={result.centroid_phi:.2f}°")
    print(f"Reconstructed θ={recovered_theta:.2f}°, φ={recovered_phi:.2f}°")
    print(f"Angular separation: {separation:.2f}°")
    print(f"Retained directions: {result.retained.sum()} of {len(result.scores)}")

    fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(8, 6.3))
    fig.subplots_adjust(left=.20, right=.80, top=.78, bottom=.21)
    figure_header(
        fig, f"The reconstructed direction is {separation:.2f}° from the injection",
        "Detector coordinates · radial distance = zenith · polar angle = azimuth",
        "Source: Crystal Eye detector response templates · markers do not define a confidence region.",
    )
    t, p, score = result.scores[result.retained].T
    ax.scatter(np.deg2rad(p), t, s=16 + 62 * score / score.max(), color=TEAL,
               alpha=.45, label="Retained templates")
    ax.scatter(np.deg2rad(phi_deg), theta_deg, s=95, facecolors="none",
               edgecolors=INK, linewidths=1.4, label="Injected")
    ax.scatter(np.deg2rad(recovered_phi), recovered_theta, marker="+", s=125,
               color=AMBER, linewidths=2, label="Reconstructed")
    ax.set_rmax(max(15, theta_deg + 5, recovered_theta + 5, float(t.max()) + 5))
    ax.set_rlabel_position(27)
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:.0f}°"))
    ax.grid(color=GRID, alpha=.75, linewidth=.8)
    ax.spines["polar"].set_color("#9CA3AF")
    fig.legend(*ax.get_legend_handles_labels(), loc="lower center", bbox_to_anchor=(.5, .09),
               ncol=3, fontsize=9)
    show_plot(fig)

## Questions to try
1. How far can you reduce the amplitude before you can no longer identify the burst by eye? Does the program's legacy trigger agree?
2. Does reducing amplitude have the same effect as changing the direction? Explain what data localization receives.
3. How close is the weighted direction to the detector centroid? Compare both with the injected direction for an off-axis example.

The preparation of the Geant4-derived inputs is documented separately in [the theory chapters](../theory/detector.md).